# MuSe corrected supplementary descriptor experiment
Run cells in order in Google Colab. This is a label-aligned descriptor classification task, **not independent emotion recognition**. The existing small HABC architecture and 10 repeated descriptor positions are retained. Scaling is fitted only on training data; validation loss selects weights. Three seeds share fixed train/validation/test partitions. Full-data medians define a fixed descriptive target (as in the original task); they are not claimed to be independently annotated outcomes. New results replace old results and are not an isolated estimate of scaling impact because validation-based stopping and training size also change.

Upload `muse_v3.csv` to Google Drive in `MyDrive/MuSe_paper2/`. Use a GPU runtime if available. Results are written to a new timestamped Drive directory. Run all cells; approve Drive mounting when prompted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
CSV_PATH = Path('/content/drive/MyDrive/MuSe_paper2/muse_v3.csv')
assert CSV_PATH.is_file(), f'Upload the CSV to {CSV_PATH}'
print('Dataset:', CSV_PATH)


In [ ]:
import os, sys, json, hashlib, platform, subprocess, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf

FEATURES = ['valence_tags','arousal_tags','dominance_tags','number_of_emotion_tags']
NAMES = ['Happy','Angry','Sad','Relaxed']
SEEDS = [42, 123, 2026]
SPLIT_SEED = 42
MAX_EPOCHS = 100
BATCH_SIZE = 64
STEPS = 10
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = Path('/content/drive/MyDrive/MuSe_paper2/results') / stamp
OUT.mkdir(parents=True, exist_ok=False)

def write_json(path, data):
    path.write_text(json.dumps(data, indent=2, allow_nan=False), encoding='utf-8')

tf.keras.utils.set_random_seed(SEEDS[0])
tf.config.experimental.enable_op_determinism()
print('TensorFlow:', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))
print('Results:', OUT)
write_json(OUT/'environment.json', {'python':sys.version,'tensorflow':tf.__version__,
    'keras':getattr(tf.keras,'__version__','unknown'),'numpy':np.__version__,
    'pandas':pd.__version__,'sklearn':sklearn.__version__, 'platform':platform.platform(),
    'gpus':[str(x) for x in tf.config.list_physical_devices('GPU')]})
(OUT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True))


In [ ]:
df = pd.read_csv(CSV_PATH)
assert set(FEATURES).issubset(df.columns), df.columns.tolist()
clean = df[FEATURES].apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
# Define the same descriptive quadrant task using the cleaned dataset's medians.
v_cut = float(clean['valence_tags'].median())
a_cut = float(clean['arousal_tags'].median())
v = clean['valence_tags'].to_numpy()
a = clean['arousal_tags'].to_numpy()
y_all = np.select([(v>=v_cut)&(a>=a_cut), (v<v_cut)&(a>=a_cut),
                   (v<v_cut)&(a<a_cut)], [0,1,2], default=3).astype('int64')
rng = np.random.RandomState(42)
chosen = []
for cls in range(4):
    ix = np.flatnonzero(y_all==cls)
    assert len(ix)>=5000, f'{NAMES[cls]} has fewer than 5000 eligible rows'
    # Same deterministic balanced selection convention as the original resample.
    chosen.extend(np.random.RandomState(42).choice(ix,5000,replace=False).tolist())
chosen = np.sort(chosen)
X = clean.iloc[chosen].to_numpy(dtype='float64')
y = y_all[chosen]
source_rows = clean.index.to_numpy()[chosen]
idx = np.arange(len(y))
trainval, test = train_test_split(idx,test_size=0.30,random_state=SPLIT_SEED,stratify=y)
train, val = train_test_split(trainval,test_size=0.20,random_state=SPLIT_SEED,stratify=y[trainval])
assert not(set(train)&set(test) or set(train)&set(val) or set(val)&set(test))
# 11200 training, 2800 validation, 6000 test for the balanced 20000-row task.
print('Train / validation / test:',len(train),len(val),len(test))
print('Label thresholds:',v_cut,a_cut)
# Quantify repeated track identifiers across partitions; do not silently claim track independence.
identity = 'lastfm_url' if 'lastfm_url' in df else None
overlap = {}
if identity:
    keys = df.loc[source_rows,identity].fillna('').astype(str).to_numpy()
    for an, ai, bn, bi in [('train',train,'test',test),('train',train,'val',val),('val',val,'test',test)]:
        overlap[f'{an}_{bn}'] = len((set(keys[ai])-{''}) & (set(keys[bi])-{''}))
    assert not any(overlap.values()), f'Track identity overlap detected: {overlap}. Stop and use a grouped split.'
manifest = pd.DataFrame({'selected_index':idx,'csv_row_index':source_rows,'label':y})
manifest['split'] = ''
for name, ii in [('train',train),('validation',val),('test',test)]: manifest.loc[ii,'split']=name
manifest.to_csv(OUT/'split_manifest.csv',index=False)
np.savez_compressed(OUT/'selected_data.npz',X=X,y=y,source_rows=source_rows,train=train,val=val,test=test)
config = {'csv_sha256':hashlib.sha256(CSV_PATH.read_bytes()).hexdigest(), 'features':FEATURES,
 'classes':NAMES,'valence_median':v_cut,'arousal_median':a_cut,
 'label_rule':'0: V>=median,A>=median; 1: V<median,A>=median; 2: V<median,A<median; 3: V>=median,A<median',
 'threshold_scope':'full cleaned dataset, fixed descriptive target',
 'selected_rows':len(y),'split_seed':SPLIT_SEED,'training_seeds':SEEDS,'steps':STEPS,
 'max_epochs':MAX_EPOCHS,'batch_size':BATCH_SIZE,'early_stopping':'val_loss, patience 10, restore best',
 'split_counts':{'train':len(train),'validation':len(val),'test':len(test)},
 'track_overlap':overlap,'interpretation':'supplementary label-aligned descriptor task; repeated static inputs'}
write_json(OUT/'config.json',config)
# A deterministic label-rule reference, not a learned-model baseline.
rule_test = np.select([(X[test,0]>=v_cut)&(X[test,1]>=a_cut),
                      (X[test,0]<v_cut)&(X[test,1]>=a_cut),
                      (X[test,0]<v_cut)&(X[test,1]<a_cut)], [0,1,2], default=3)
print('Label-rule reference accuracy:',accuracy_score(y[test],rule_test))


## Implemented model
One Conv2D(6), one BiLSTM(64 per direction), attention with position-specific bias, Dense(64), Dropout(0.3), softmax. Explicit channels-last layout preserves the original standard layout. Repetition creates no measured temporal information. Register the custom layer by executing this cell before loading a saved model.

In [ ]:
@tf.keras.utils.register_keras_serializable(package='MuSe')
class DescriptorAttention(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight',shape=(input_shape[-1],1),initializer='random_normal')
        self.b = self.add_weight(name='att_bias',shape=(input_shape[1],1),initializer='zeros')
        super().build(input_shape)
    def call(self, x):
        e = tf.squeeze(tf.tanh(tf.matmul(x,self.W)+self.b),axis=-1)
        weights = tf.nn.softmax(e,axis=1)
        return tf.reduce_sum(x*tf.expand_dims(weights,-1),axis=1)
    def get_config(self): return super().get_config()

def build_model(n_features):
    L = tf.keras.layers
    cnn = L.Input((1,STEPS,n_features),name='cnn_input')
    c = L.Conv2D(6,(3,3),padding='same',activation='relu',data_format='channels_last')(cnn)
    c = L.MaxPooling2D((1,2),strides=(1,2),padding='same',data_format='channels_last')(c)
    c = L.Flatten()(c)
    seq = L.Input((STEPS,n_features),name='lstm_input')
    h = L.Bidirectional(L.LSTM(64,return_sequences=True))(seq)
    h = DescriptorAttention()(h)
    z = L.Concatenate()([c,h])
    z = L.Dense(64,activation='relu')(z)
    z = L.Dropout(0.3)(z)
    out = L.Dense(4,activation='softmax')(z)
    model = tf.keras.Model([cnn,seq],out)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                  loss='categorical_crossentropy',metrics=['accuracy'])
    return model

def encode(z):
    seq = np.repeat(z[:,None,:],STEPS,axis=1).astype('float32')
    return [seq[:,None,:,:],seq]


In [ ]:
# Six fits: two feature groups x three seeds. Test results never select weights or seeds.
GROUPS = {'all_features':list(range(4)), 'sentiment_only':[0,1,2]}
records = []
for group, cols in GROUPS.items():
    group_dir = OUT/group
    group_dir.mkdir(exist_ok=True)
    scaler = StandardScaler().fit(X[train][:,cols])
    joblib.dump(scaler,group_dir/'training_scaler.joblib')
    train_inputs = encode(scaler.transform(X[train][:,cols]))
    val_inputs = encode(scaler.transform(X[val][:,cols]))
    test_inputs = encode(scaler.transform(X[test][:,cols]))
    for seed in SEEDS:
        run = group_dir/f'seed_{seed}'
        run.mkdir(exist_ok=True)
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        model = build_model(len(cols))
        (run/'architecture.json').write_text(model.to_json())
        with (run/'model_summary.txt').open('w') as f:
            model.summary(print_fn=lambda line: f.write(line+'\n'))
        callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True),
                     tf.keras.callbacks.ModelCheckpoint(str(run/'best_model.keras'),monitor='val_loss',save_best_only=True),
                     tf.keras.callbacks.CSVLogger(str(run/'epoch_log.csv')),
                     tf.keras.callbacks.TerminateOnNaN()]
        print('Training',group,'seed',seed)
        hist = model.fit(train_inputs,tf.keras.utils.to_categorical(y[train],4),
            validation_data=(val_inputs,tf.keras.utils.to_categorical(y[val],4)),
            epochs=MAX_EPOCHS,batch_size=BATCH_SIZE,callbacks=callbacks,verbose=2)
        assert np.isfinite(hist.history['val_loss']).all(), 'Non-finite validation loss; stop.'
        # Explicitly load the validation-selected checkpoint before the single final test evaluation.
        model = tf.keras.models.load_model(run/'best_model.keras')
        prob = model.predict(test_inputs,verbose=0)
        pred = prob.argmax(axis=1)
        np.savez_compressed(run/'test_predictions.npz',y_true=y[test],y_pred=pred,
                            probabilities=prob,source_rows=source_rows[test])
        report = classification_report(y[test],pred,labels=list(range(4)),target_names=NAMES,
                                       output_dict=True,zero_division=0)
        write_json(run/'classification_report.json',report)
        pd.DataFrame(report).T.to_csv(run/'classification_report.csv')
        metrics = {'group':group,'seed':seed,'accuracy':float(accuracy_score(y[test],pred)),
          'macro_precision':report['macro avg']['precision'],'macro_recall':report['macro avg']['recall'],
          'macro_f1':report['macro avg']['f1-score'],'weighted_f1':report['weighted avg']['f1-score'],
          'epochs_run':len(hist.history['loss']),
          'best_validation_epoch':int(np.argmin(hist.history['val_loss'])+1)}
        write_json(run/'metrics.json',metrics)
        cm = confusion_matrix(y[test],pred,labels=list(range(4)))
        np.save(run/'confusion_counts.npy',cm)
        fig,ax=plt.subplots(figsize=(6,5))
        ConfusionMatrixDisplay.from_predictions(y[test],pred,labels=list(range(4)),
            display_labels=NAMES,normalize='true',values_format='.3f',ax=ax,colorbar=False)
        ax.set_title(f'{group}; seed {seed}; descriptor task')
        fig.tight_layout(); fig.savefig(run/'confusion_normalized.png',dpi=200); plt.close(fig)
        fig,axs=plt.subplots(1,2,figsize=(10,4))
        for key,ax in [('loss',axs[0]),('accuracy',axs[1])]:
            ax.plot(hist.history[key],label='training'); ax.plot(hist.history['val_'+key],label='validation')
            ax.set_xlabel('Epoch'); ax.set_ylabel(key); ax.legend()
        fig.tight_layout(); fig.savefig(run/'training_curves.png',dpi=200); plt.close(fig)
        records.append(metrics)
        pd.DataFrame(records).to_csv(OUT/'run_metrics.csv',index=False)
        print(metrics)


In [ ]:
results = pd.read_csv(OUT/'run_metrics.csv')
metrics = ['accuracy','macro_precision','macro_recall','macro_f1','weighted_f1']
summary = results.groupby('group')[metrics].agg(['mean','std'])
summary.to_csv(OUT/'summary_mean_sd.csv')
print('Percent values; SD is across three training seeds on fixed partitions:')
display((summary*100).round(3))
# Use this predeclared seed for the illustrative full-feature class report, not the best test seed.
print('Illustrative class report: all_features / seed 42')
display(pd.read_csv(OUT/'all_features/seed_42/classification_report.csv',index_col=0))
write_json(OUT/'completion.json',{'complete':len(results)==len(SEEDS)*len(GROUPS),
    'completed_runs':len(results),'output_directory':str(OUT)})
archive = shutil.make_archive('/content/muse_corrected_'+stamp,'zip',root_dir=str(OUT))
print('All results are saved in Drive:',OUT)
print('Downloadable archive:',archive)
from google.colab import files
files.download(archive)


## Reporting the rerun
Use `summary_mean_sd.csv` for the seed-averaged feature comparison and `all_features/seed_42/` for the predeclared illustrative confusion matrix and class report. Do not combine a mean accuracy with a single-run confusion matrix as if they were one result. Replace all old MuSe numbers and figures consistently. Report 11200 training / 2800 validation / 6000 test items, training-only scaling, validation-loss stopping, seeds and environment versions. High scores remain evidence only of label-aligned descriptor classification. No cross-subject or general music-emotion claim follows. Keep this notebook alongside the exported results.